In [ ]:
!pip -q install rasterio
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/cubesat_scenes
!mkdir -p masks

In [ ]:
from glob import glob
import warnings
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from scipy import ndimage

warnings.filterwarnings('ignore', category=RuntimeWarning)

KEY = 'orel_2025'
NDVI_MAX = 0.5
NDVI_AMP = 0.25
MIN_PX = 20
BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'NDVI', 'NDBI', 'NDMI']

paths = sorted(glob(f'scenes/{KEY}-*.tif'))
print(paths)

In [ ]:
def read(path):
    with rasterio.open(path) as src:
        a = src.read().astype('float32')
        profile = src.profile
    a[a == -9999] = np.nan
    return dict(zip(BANDS, a)), profile

data = [read(p) for p in paths]
scenes = [d[0] for d in data]
profile = data[0][1]
print(len(scenes), 'снимка,', profile['width'], 'x', profile['height'], 'пикселей,', profile['crs'])

In [ ]:
def rgb(s):
    img = np.dstack([s['B4'], s['B3'], s['B2']])
    return np.clip(img / np.nanpercentile(img, 98), 0, 1)

In [ ]:
s = scenes[2]
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
for ax, name in zip(axes.ravel(), BANDS):
    if name.startswith('B'):
        ax.imshow(s[name], cmap='gray')
    else:
        ax.imshow(s[name], cmap='RdYlGn', vmin=-1, vmax=1)
    ax.set_title(name)
    ax.axis('off')
plt.show()

In [ ]:
fig, axes = plt.subplots(2, len(scenes), figsize=(6 * len(scenes), 12), squeeze=False)
for j, (p, s) in enumerate(zip(paths, scenes)):
    axes[0, j].imshow(rgb(s))
    axes[0, j].set_title(p.split('-', 1)[1])
    axes[1, j].imshow(s['NDVI'], cmap='RdYlGn', vmin=-1, vmax=1)
    axes[1, j].set_title('NDVI')
for ax in axes.ravel():
    ax.axis('off')
plt.show()

# Формула маски
Маска полей по формуле
$$
\text{поле} = \left(\mathrm{NDVI}_{\max} > 0{,}5\right) \;\ and \; \left(\mathrm{NDVI}_{\max} - \mathrm{NDVI}_{\min} > 0{,}25\right) \;\ and \; \ not \left(\mathrm{NDBI}_{\min} > 0\right)
$$

## Где

$$
\mathrm{NDVI} = \frac{B5 - B4}{B5 + B4}, \qquad \mathrm{NDBI} = \frac{B6 - B5}{B6 + B5}
$$

$\mathrm{NDVI}_{\max}$, $\mathrm{NDVI}_{\min}$, $\mathrm{NDBI}_{\min}$ — максимум и минимум по датам сезона для каждого пикселя.

- $\mathrm{NDVI}_{\max} > 0{,}5$ — пиксель хотя бы раз был густо-зелёным (отсекает воду, город, дороги)
- $\mathrm{NDVI}_{\max} - \mathrm{NDVI}_{\min} > 0{,}25$ — зелёность сильно менялась за сезон (отсекает лес и луга)
- $\neg(\mathrm{NDBI}_{\min} > 0)$ — пиксель не выглядел застройкой во все даты


In [ ]:
ndvi = np.stack([s['NDVI'] for s in scenes])
ndbi = np.stack([s['NDBI'] for s in scenes])
hi = np.nanmax(ndvi, 0)
lo = np.nanmin(ndvi, 0)
built = np.nanmin(ndbi, 0)
raw = (hi > NDVI_MAX) & (hi - lo > NDVI_AMP) & ~(built > 0)

fig, axes = plt.subplots(1, 4, figsize=(24, 6))
axes[0].imshow(hi, cmap='RdYlGn', vmin=-1, vmax=1); axes[0].set_title('NDVI max')
axes[1].imshow(hi - lo, cmap='magma', vmin=0, vmax=1); axes[1].set_title('NDVI max − min')
axes[2].imshow(built, cmap='RdBu_r', vmin=-0.5, vmax=0.5); axes[2].set_title('NDBI min')
axes[3].imshow(raw, cmap='gray'); axes[3].set_title('по формуле')
for ax in axes:
    ax.axis('off')
plt.show()

# Обоснование порогов

**Источники**
1. Барталёв С.А., Егоров В.А., Лупян Е.А., Плотников Д.Е., Уваров И.А. Распознавание пахотных земель на основе многолетних спутниковых данных спектрорадиометра MODIS и локально-адаптивной классификации // Компьютерная оптика. 2011. Т. 35, № 1. С. 103–116 ([PDF](https://computeroptics.ru/KO/PDF/KO35-1/13.pdf)). ИКИ РАН, метод карты пахотных земель всей России.
2. Филиппов Д.В., Тарнопольский Л.А., Бояренкова А.Д. Исследование влияния освещенности на расчет вегетационных индексов на примере NDVI // Вестник СГУГиТ. 2023. Т. 28, № 5. С. 151–162. DOI [10.33764/2411-1759-2023-28-5-151-162](https://vestnik.sgugit.ru/upload/vestnik/sborniki/2023/28_5/151-162.pdf). НГИЦ РАН.
3. Черепанов А.С., Дружинина Е.Г. Спектральные свойства растительности и вегетационные индексы // Геоматика. 2009. № 3. С. 28–32 ([PDF](https://sovzond.ru/upload/iblock/3b8/2009_03_005.pdf)).

**Почему нужны несколько дат.** По одному снимку пашню «часто можно ошибочно отнести к классу открытых почв или лугов (степей)» [1, с. 108]. Поэтому маска строится по сезонному ряду NDVI: максимум и минимум за май–август.

**NDVI_max > 0,5.** Градации NDVI [2, с. 152]: «от 0,1 до 0,2 чаще соответствуют открытой почве», «от 0,2 до 0,4 — слабая, разреженная растительность, от 0,4 до 0,6 — умеренная растительность и выше 0,6 — здоровая, густая растительность»; для зелёной растительности NDVI обычно 0,2–0,8 [3, с. 32]. Порог 0,5 — середина класса «умеренная»: пиксель хотя бы в одну дату был заметно зелёным. Почва (≤ 0,2) и разреженная растительность (≤ 0,4) отсекаются с запасом не меньше 0,1, а у посевов на пике NDVI выше 0,6. Лес этот критерий тоже проходит, поэтому нужен второй.

**NDVI_max − NDVI_min > 0,25.**
- *Признак.* В методе ИКИ РАН есть два близких признака пашни [1, с. 109–110, табл. 1]. «Индекс сезонного снижения фитомассы» — сезонные минимумы с 15 мая по 15 сентября. На пашне минимум низкий из-за «осенней обработки почвы, скашивания многолетних трав или предпосевных работ», а «естественная растительность накапливает существенный объём фитомассы». «Разностный индекс сезонного пика» — разность максимума и среднего: для пашни «характерны более существенные различия между максимальными и среднегодовыми значениями вегетационного индекса» «в силу использования агротехники и ускоренного развития растительности». Наш размах «максимум − минимум» — тот же признак по снимкам Landsat за один сезон.
- *Нижняя граница для пашни.* Пашня проходит путь «почва (0,1–0,2) → посев (> 0,5 по первому критерию)» [2], значит её размах не меньше 0,5 − 0,2 = 0,3.
- *Верхняя граница для леса.* По снимкам Sentinel-2 одного участка с разницей в два дня (29.08.2021, облачность 31,4 %, и 31.08.2021, облачность 3,4 %) половина пикселей леса перешла из класса 0,6–0,8 в класс 0,8–1,0, а основная часть пашни — из класса 0,2–0,4 в класс 0,4–0,6 [2, табл. 4]. Значит, одна только освещённость сдвигает NDVI примерно на один класс, 0,2. Порог размаха должен быть больше 0,2, иначе лес пройдёт как пашня из-за разной освещённости дат.
- *Итог.* 0,25 лежит между шумом освещённости (≈ 0,2) и минимальным размахом пашни (≈ 0,3). При NDVI_max около 0,5 условие равносильно «минимум ниже половины сезонного максимума» — та же половина максимума, что в «индексе кратчайшего сезона вегетации» ИКИ РАН [1, табл. 1].

В источниках 1–3 нет готового числа 0,25: ИКИ РАН подбирает границы обучаемым классификатором по опорным данным [1]. Поэтому ниже порог дополнительно проверяется на данных: маска сравнивается с классом «пашня» (значение 40) карты ESA WorldCover 2021 ([каталог Earth Engine](https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v200)). Файл `worldcover-orel_2025.tif` выгружает `gee/export.js` на ту же сетку, что и снимки.

Ниже: (1) распределения NDVI_max и размаха для пашни, леса и лугов — пороги должны лежать между классами; (2) перебор пар порогов и F1 — пара (0,5; 0,25) должна быть близка к лучшей.

In [ ]:
with rasterio.open(f'worldcover-{KEY}.tif') as src:
    wc = src.read(1)
ok = np.isfinite(hi) & np.isfinite(lo)
classes = {'пашня (40)': 40, 'лес (10)': 10, 'луга (30)': 30}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for name, v in classes.items():
    sel = ok & (wc == v)
    axes[0].hist(hi[sel], bins=60, range=(-0.2, 1), alpha=0.5, density=True, label=f'{name}, {sel.sum()} пикс.')
    axes[1].hist((hi - lo)[sel], bins=60, range=(0, 1), alpha=0.5, density=True, label=name)
axes[0].axvline(NDVI_MAX, color='k', ls='--'); axes[0].set_title('NDVI max')
axes[1].axvline(NDVI_AMP, color='k', ls='--'); axes[1].set_title('NDVI max − min')
for ax in axes:
    ax.legend()
plt.show()

for name, v in classes.items():
    sel = ok & (wc == v)
    print(f'{name}: NDVI max > {NDVI_MAX} — {(hi[sel] > NDVI_MAX).mean():.0%},  размах > {NDVI_AMP} — {((hi - lo)[sel] > NDVI_AMP).mean():.0%}')

In [ ]:
ref = (wc == 40)[ok]
maxs = np.round(np.arange(0.3, 0.81, 0.05), 2)
amps = np.round(np.arange(0.05, 0.51, 0.05), 2)
f1 = np.zeros((len(maxs), len(amps)))
for i, t_max in enumerate(maxs):
    for j, t_amp in enumerate(amps):
        pred = ((hi > t_max) & (hi - lo > t_amp) & ~(built > 0))[ok]
        tp, fp, fn = (pred & ref).sum(), (pred & ~ref).sum(), (~pred & ref).sum()
        f1[i, j] = 2 * tp / (2 * tp + fp + fn)

i, j = np.unravel_index(f1.argmax(), f1.shape)
ours = f1[list(maxs).index(NDVI_MAX), list(amps).index(NDVI_AMP)]
print(f'лучшая пара: NDVI max > {maxs[i]}, размах > {amps[j]}, F1 = {f1[i, j]:.3f}')
print(f'наша пара:   NDVI max > {NDVI_MAX}, размах > {NDVI_AMP}, F1 = {ours:.3f}')

pred = ((hi > NDVI_MAX) & (hi - lo > NDVI_AMP) & ~(built > 0))[ok]
tp, fp, fn = (pred & ref).sum(), (pred & ~ref).sum(), (~pred & ref).sum()
print(f'precision {tp / (tp + fp):.3f}, recall {tp / (tp + fn):.3f} относительно WorldCover')

plt.figure(figsize=(9, 7))
plt.imshow(f1, cmap='viridis', origin='lower')
plt.xticks(range(len(amps)), amps); plt.yticks(range(len(maxs)), maxs)
plt.xlabel('порог размаха NDVI'); plt.ylabel('порог NDVI max')
plt.plot(list(amps).index(NDVI_AMP), list(maxs).index(NDVI_MAX), 'r+', ms=20, mew=3, label='наши пороги')
plt.plot(j, i, 'wx', ms=14, mew=3, label='максимум F1')
plt.colorbar(label='F1 по WorldCover «пашня»'); plt.legend()
plt.show()

WorldCover — карта за 2021 год, построенная моделью, это опорные данные, а не наземная истина: часть расхождений — ошибки самой карты и смена культур между 2021 и 2025 годами. Если максимум F1 лежит далеко от (0,5; 0,25), пороги `NDVI_MAX` и `NDVI_AMP` в начале ноутбука стоит заменить на найденные.

# Очистка

In [ ]:
opened = ndimage.binary_opening(raw, np.ones((3, 3)))
labels, n = ndimage.label(opened)
keep = np.bincount(labels.ravel()) >= MIN_PX
keep[0] = False
mask = keep[labels]
print(f'пятен {n}, оставлено {keep.sum()}, поля {mask.mean():.1%} площади')

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, img, title in zip(axes, [raw, opened, mask], ['по формуле', 'после открытия', 'без мелких пятен']):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.show()

# Вывод с маской и сохранение

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(rgb(scenes[2]))
plt.imshow(np.ma.masked_where(~mask, mask), cmap='autumn', alpha=0.4)
plt.axis('off')
plt.show()

In [ ]:
with rasterio.open(f'masks/{KEY}.tif', 'w', **dict(profile, count=1, dtype='uint8', nodata=None)) as dst:
    dst.write(mask.astype('uint8'), 1)
print('сохранено', f'masks/{KEY}.tif')